[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/90_exercises.ipynb)

# Bài tập — Chương 23: Probabilistic Machine Learning (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

warnings.filterwarnings("ignore", category=ConvergenceWarning)
rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Công thức trung bình và hiệp phương sai của $p(\mathbf x_a \mid \mathbf x_b)$ với Gaussian chung. (b) Câu chuyện sinh dữ liệu của GMM; E-step và M-step. (c) Posterior của hồi quy tuyến tính Bayes ($\mathbf m_N$, $S_N$) và phương sai dự đoán. (d) Định nghĩa Gaussian Process. (e) Ba bài toán của HMM và thuật toán cho từng bài.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Đồng nghiệp dùng Gaussian Process (kernel RBF) dự báo doanh số 6 tháng tới từ 3 năm dữ liệu có xu hướng tăng. Báo cáo viết: *"Model dự báo doanh số sẽ dần quay về mức trung bình lịch sử; khoảng tin cậy 95% của tháng 1 rất hẹp nên dự báo tháng 1 rất đáng tin."* Nhận xét hai kết luận này. Bạn kiểm tra và cải thiện thế nào?

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** $\boldsymbol\mu = [1, 2]$, $\Sigma = \begin{bmatrix}4 & 2\\2 & 3\end{bmatrix}$. Quan sát $x_2 = 5$. Tính $\mathbb E[x_1 \mid x_2 = 5]$, $\text{Var}(x_1 \mid x_2 = 5)$ và hệ số tương quan $\rho$. So sánh phương sai có điều kiện với $\sigma_1^2(1 - \rho^2)$.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
mean3 = None
var3 = None
rho3 = None
var3_check = None
### END CODE HERE ###
print(f"E = {mean3:.4f} | Var = {var3:.4f} | ρ = {rho3:.4f} | σ₁²(1 − ρ²) = {var3_check:.4f}")

In [ ]:
check_close(mean3, 3.0, message="kỳ vọng có điều kiện")
check_close(var3, 8 / 3, message="phương sai có điều kiện")
check_close(var3, var3_check, message="Var(x₁|x₂) = σ₁²(1 − ρ²)")

**Câu 4.** (a) GMM 1D hai thành phần: $\pi = [0.5, 0.5]$, $\mu = [0, 4]$, $\sigma = [1, 1]$. Responsibility của thành phần 1 cho điểm $x = 1.5$?
(b) Hồi quy Bayes 1 trọng số, $y = wx + \varepsilon$: prior $w \sim \mathcal N(0, 1)$ ($\alpha = 1$), $\beta = 4$. Dữ liệu $x = [1, 2]$, $y = [1, 3]$. Tính $S_N$, $m_N$, và phương sai dự đoán tại $x^* = 3$.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
r4 = None
S_N4 = None
m_N4 = None
pred_var4 = None
### END CODE HERE ###
print(f"r = {r4:.4f} | S_N = {S_N4:.4f} | m_N = {m_N4:.4f} | Var dự đoán tại x=3: {pred_var4:.4f}")

In [ ]:
check_close(r4, 1 / (1 + np.exp(-2)), tol=1e-9, message="responsibility = sigmoid(2)")
check_close(S_N4, 1 / 21, message="S_N")
check_close(m_N4, 28 / 21, message="m_N")
check_close(pred_var4, 0.25 + 9 / 21, message="phương sai dự đoán = nhiễu + bất định tham số")

## L3 — Code

**Bài 5.** Viết `conditional_gaussian(mu, Sigma, idx_a, idx_b, x_b)` → `(mean, cov)` của $p(\mathbf x_a \mid \mathbf x_b)$ (dùng `np.linalg.solve`, không nghịch đảo). Kiểm tra bằng Monte Carlo trên Gaussian 3 chiều, và với trường hợp $\mathbf x_b$ độc lập với $\mathbf x_a$.

In [ ]:
def conditional_gaussian(mu, Sigma, idx_a, idx_b, x_b):
    ### START CODE HERE ### (≈ 5 dòng)
    # S_aa, S_ab, S_bb bằng Sigma[np.ix_(...)]; K = S_ab S_bb⁻¹ (= solve(S_bb, S_abᵀ)ᵀ)
    # mean = mu_a + K (x_b − mu_b); cov = S_aa − K S_ba
    pass
    ### END CODE HERE ###

In [ ]:
mu5 = np.array([0.0, 1.0, -1.0])
S5 = np.array([[2.0, 0.8, 0.6], [0.8, 1.0, 0.3], [0.6, 0.3, 1.5]])
m5, c5 = conditional_gaussian(mu5, S5, [0], [1, 2], [2.0, 0.0])
samples = rng.multivariate_normal(mu5, S5, 2_000_000)
near = samples[(np.abs(samples[:, 1] - 2.0) < 0.05) & (np.abs(samples[:, 2] - 0.0) < 0.05), 0]
print(f"công thức: mean = {m5[0]:.3f}, var = {c5[0, 0]:.3f} | Monte Carlo ({len(near)} mẫu): mean = {near.mean():.3f}, var = {near.var():.3f}")
check_close(m5[0], near.mean(), tol=0.05, message="kỳ vọng khớp Monte Carlo")
check_close(c5[0, 0], near.var(), tol=0.08, message="phương sai khớp Monte Carlo")
m_ind, c_ind = conditional_gaussian([0, 0], np.diag([2.0, 3.0]), [0], [1], [10.0])
check_close(m_ind, [0.0], message="độc lập → quan sát x_b không đổi kỳ vọng")
check_close(c_ind, [[2.0]], message="độc lập → phương sai giữ nguyên")

**Bài 6.** Viết `em_gmm_1d(x, K, n_iter, seed)` cho GMM 1 chiều → `(weights, means, variances, history)`. Khởi tạo: means = K điểm ngẫu nhiên của `x`, variances = var(x), weights đều. E-step trong không gian log. Kiểm tra: log-likelihood tăng đơn điệu và kết quả khớp `GaussianMixture`.

In [ ]:
def em_gmm_1d(x, K, n_iter=300, seed=0, tol=1e-10):
    r = np.random.default_rng(seed)
    means = r.choice(x, K, replace=False).astype(float)
    variances = np.full(K, x.var())
    weights = np.full(K, 1 / K)
    history = []
    for _ in range(n_iter):
        ### START CODE HERE ### (≈ 9 dòng)
        # E-step: log_p[i, k] = log weights[k] + norm.logpdf(x[i], means[k], sqrt(variances[k])); chuẩn hoá bằng logsumexp → R
        # lưu log-likelihood = tổng logsumexp vào history
        # M-step: Nk = R.sum(0); weights, means, variances có trọng số (+ 1e-9 cho variances)
        pass
        ### END CODE HERE ###
        if len(history) > 1 and abs(history[-1] - history[-2]) < tol * abs(history[-1]):
            break
    return weights, means, variances, np.array(history)

In [ ]:
x6 = np.r_[rng.normal(-2, 0.5, 300), rng.normal(1, 1.0, 500), rng.normal(5, 0.7, 200)]
w6, m6, v6, h6 = em_gmm_1d(x6, 3, seed=1)
order = np.argsort(m6)
print("weights:", w6[order].round(3), "| means:", m6[order].round(3), "| std:", np.sqrt(v6[order]).round(3), f"| {len(h6)} vòng")
check_true(np.all(np.diff(h6) > -1e-8), "log-likelihood không giảm")
sk6 = GaussianMixture(3, n_init=5, random_state=0).fit(x6[:, None])
check_close(h6[-1] / len(x6), sk6.score(x6[:, None]), tol=1e-3, message="log-lik/điểm khớp sklearn")
check_close(m6[order], np.sort(sk6.means_.ravel()), tol=0.02, message="means khớp sklearn")

## L4 — Debug

**Bài 7.** Hàm GP regression dưới đây có 3 lỗi: (1) kernel RBF dùng $\exp(-d^2/\ell)$ thay vì $\exp(-d^2/(2\ell^2))$; (2) quên cộng phương sai nhiễu $\sigma_n^2 I$ vào $K$; (3) phương sai hậu nghiệm **cộng** thay vì **trừ** $\mathbf v^\top\mathbf v$. Viết lại `gp_posterior` đúng (dùng Cholesky) và so với sklearn (kernel cố định).

In [ ]:
def rbf_kernel(a, b, ell, sf):
    ### START CODE HERE ### (≈ 1 dòng)
    return sf**2 * np.exp(-(a[:, None] - b[None]) ** 2 / ell)      # 🐛 (1)
    ### END CODE HERE ###


def gp_posterior(X, y, X_star, ell=1.0, sf=1.0, noise=0.1):
    ### START CODE HERE ### (≈ 7 dòng)
    K = rbf_kernel(X, X, ell, sf) + 1e-10 * np.eye(len(X))                  # 🐛 (2)
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    K_s = rbf_kernel(X, X_star, ell, sf)
    v = np.linalg.solve(L, K_s)
    return K_s.T @ alpha, sf**2 + np.sum(v**2, axis=0)                      # 🐛 (3)
    ### END CODE HERE ###

In [ ]:
X7 = np.r_[rng.uniform(-3, 3, 25), [0.5, 0.5]]                         # có điểm trùng lặp
y7 = np.sin(X7) + rng.normal(0, 0.1, len(X7))
xs7 = np.linspace(-4, 4, 50)
m7, v7 = gp_posterior(X7, y7, xs7, ell=0.8, sf=1.2, noise=0.1)
ref7 = GaussianProcessRegressor(ConstantKernel(1.44, "fixed") * RBF(0.8, "fixed"), alpha=0.01, optimizer=None).fit(X7[:, None], y7)
mr7, sr7 = ref7.predict(xs7[:, None], return_std=True)
check_close(m7, mr7, tol=1e-8, message="trung bình khớp sklearn")
check_close(np.sqrt(np.maximum(v7, 0)), sr7, tol=1e-6, message="độ lệch chuẩn khớp sklearn")
check_true(np.all(v7 <= 1.44 + 1e-9), "phương sai hậu nghiệm ≤ phương sai prior")

## L5 — Phân tích

**Bài 8.** Dữ liệu `customer_segments.csv` (4 feature, 5 phân khúc thật trong cột `true_segment` — **chỉ dùng để đánh giá**). Sau `StandardScaler`, khảo sát GMM với $K \in \{2, \dots, 8\}$ và `covariance_type` ∈ {full, tied, diag, spherical} (`n_init=5`, `random_state=0`). Lưu BIC vào DataFrame `bic8` (hàng K, cột kiểu); `best_k8`, `best_cov8` theo BIC nhỏ nhất; `ari_gmm8` của model tốt nhất và `ari_km8` của K-Means cùng K. Nhận xét.

In [ ]:
seg = pd.read_csv(repo_path("data", "sample", "customer_segments.csv"))
X8 = StandardScaler().fit_transform(seg[["annual_income_musd", "spending_score", "age", "visits_per_month"]])
y8 = seg["true_segment"]
### START CODE HERE ### (≈ 10 dòng)
bic8 = None       # DataFrame: index K = 2..8, cột full/tied/diag/spherical
best_k8, best_cov8 = None, None
ari_gmm8, ari_km8 = None, None
### END CODE HERE ###
print(bic8.round(0))
print(f"tốt nhất theo BIC: K = {best_k8}, {best_cov8} | ARI GMM = {ari_gmm8:.3f} | ARI K-Means = {ari_km8:.3f}")

In [ ]:
check_equal(int(best_k8), 5, "BIC tìm đúng 5 phân khúc")
check_greater(ari_gmm8, 0.8, "GMM khớp phân khúc thật")
check_true(bic8.shape == (7, 4), "đủ 7 × 4 cấu hình")

## L6 — Thực tế

**Bài 9.** **Bayesian optimization** cho siêu tham số (chương 16). Hàm mục tiêu: accuracy cross-validation (3-fold) của `SVC(C=10, gamma=10**t)` trên 600 ảnh digits đầu, với $t = \log_{10}\gamma \in [-5, 3]$ — chỉ một dải hẹp của $t$ cho kết quả tốt. Giả sử mỗi lần đánh giá rất "đắt" → chỉ được **8 lần**.
- Bắt đầu với 3 điểm $t \in \{-5, -1, 3\}$; mỗi vòng: fit `GaussianProcessRegressor` (ConstantKernel × RBF + WhiteKernel, `normalize_y=True`) trên các điểm đã có, chọn điểm tiếp theo **chưa thử** trên lưới 161 điểm tối đa **Expected Improvement**:
$$\text{EI}(t) = (\mu - f^* - \xi)\,\Phi(z) + \sigma\,\phi(z), \quad z = \frac{\mu - f^* - \xi}{\sigma}, \quad \xi = 0.01$$
- So với **random search** 8 điểm $t \sim U(-5, 3)$ (20 lần lặp, seed 0–19): `best_bo9`, `best_random9` (list 20 giá trị), `best_grid9` (lưới 161 điểm — chỉ để tham chiếu, thực tế không đủ ngân sách).

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd, yd = Xd[:600] / 16, yd[:600]
cache9 = {}


def objective(t):
    t = round(float(t), 6)
    if t not in cache9:
        cache9[t] = cross_val_score(SVC(C=10, gamma=10**t), Xd, yd, cv=3).mean()
    return cache9[t]


grid9 = np.linspace(-5, 3, 161)
best_grid9 = max(objective(t) for t in grid9)


def expected_improvement(mu, sigma, f_best, xi=0.01):
    ### START CODE HERE ### (≈ 3 dòng)
    return None   # z = (mu − f_best − xi) / sigma; EI theo công thức (sigma = 0 → EI = 0)
    ### END CODE HERE ###


ts = [-5.0, -1.0, 3.0]
scores = [objective(t) for t in ts]
### START CODE HERE ### (≈ 9 dòng)
# lặp tới khi đủ 8 điểm: fit GP trên (ts, scores) → mu, sigma trên grid9 → EI; gán EI = −inf cho điểm đã thử
# → chọn argmax EI → đánh giá, thêm vào ts/scores
best_bo9 = None
best_random9 = None   # list: với mỗi seed 0..19, max objective tại 8 điểm U(−5, 3)
### END CODE HERE ###
print(f"điểm BO đã thử (log10 γ): {np.round(ts, 2)}")
print(f"accuracy tốt nhất: BO (8 lần) = {best_bo9:.4f} | random (8 lần): TB = {np.mean(best_random9):.4f}, tệ nhất = {min(best_random9):.4f} | "
      f"lưới 161 điểm = {best_grid9:.4f}")
print(f"BO tốt hơn hoặc bằng random search ở {np.mean(best_bo9 >= np.array(best_random9)):.0%} số lần lặp")

In [ ]:
check_greater(best_bo9, best_grid9 - 0.01, "BO tìm gần tối ưu chỉ với 8 lần đánh giá")
check_greater(best_bo9, np.mean(best_random9) + 0.005, "BO tốt hơn rõ random search trung bình")
check_equal(len(set(np.round(ts, 6))), 8, "không đánh giá lặp lại một điểm")
check_close(expected_improvement(np.array([1.0]), np.array([0.0]), 2.0), [0.0], tol=1e-9, message="EI = 0 khi chắc chắn tệ hơn")

## L7 — Nâng cao

**Bài 10.** **HMM với phát xạ Gaussian — phát hiện chế độ thị trường.** Lợi suất ngày sinh từ 2 chế độ: *bình lặng* ($\mu = 0.05\%$, $\sigma = 0.8\%$) và *biến động* ($\mu = -0.1\%$, $\sigma = 2.5\%$); xác suất chuyển bình lặng→biến động 0.02, biến động→bình lặng 0.05.

Cài đặt `gaussian_hmm_fit(x, K, n_iter, seed)` (Baum–Welch): khởi tạo means bằng phân vị của x, std bằng std(x)·[0.5, 1.5], A = 0.9 trên đường chéo; E-step forward–backward trong không gian log với $\log B_t(k) = \log\mathcal N(x_t \mid \mu_k, \sigma_k^2)$; M-step: $\boldsymbol\pi = \gamma_1$, $A \propto \sum_t\xi_t$, $\mu_k, \sigma_k^2$ = trung bình/phương sai có trọng số $\gamma_t(k)$. Trả về `(pi, A, mu, sigma, gamma, history)`. Sau đó gán chế độ bằng $\arg\max_k\gamma_t(k)$ và đo độ chính xác (`acc10`).

In [ ]:
def simulate_regimes(T, seed):
    r = np.random.default_rng(seed)
    A = np.array([[0.98, 0.02], [0.05, 0.95]])
    mu, sd = np.array([0.0005, -0.001]), np.array([0.008, 0.025])
    z = np.empty(T, int)
    z[0] = 0
    for t in range(1, T):
        z[t] = r.choice(2, p=A[z[t - 1]])
    return z, r.normal(mu[z], sd[z])


def gaussian_hmm_fit(x, K=2, n_iter=200, seed=0, tol=1e-8):
    T = len(x)
    pi = np.full(K, 1 / K)
    A = np.full((K, K), 0.1 / (K - 1)) + np.eye(K) * (0.9 - 0.1 / (K - 1))
    mu = np.quantile(x, np.linspace(0.3, 0.7, K))
    sigma = x.std() * np.linspace(0.5, 1.5, K)
    history = []
    for _ in range(n_iter):
        ### START CODE HERE ### (≈ 22 dòng)
        # log_B[t, k] = norm.logpdf(x[t], mu[k], sigma[k])
        # forward: log_alpha[0] = log pi + log_B[0]; log_alpha[t] = logsumexp(log_alpha[t−1][:, None] + log A, axis=0) + log_B[t]
        # backward: log_beta[T−1] = 0; log_beta[t] = logsumexp(log A + log_B[t+1] + log_beta[t+1], axis=1)
        # ll = logsumexp(log_alpha[−1]); history.append(ll)
        gamma = None   # exp(log_alpha + log_beta − ll), shape (T, K)
        # xi[t, i, j] = exp(log_alpha[t, i] + log A[i, j] + log_B[t+1, j] + log_beta[t+1, j] − ll)
        # M-step: pi = gamma[0]; A = xi.sum(0) chuẩn hoá theo hàng; mu, sigma có trọng số gamma
        ### END CODE HERE ###
        if len(history) > 1 and abs(history[-1] - history[-2]) < tol * abs(history[-1]):
            break
    return pi, A, mu, sigma, gamma, np.array(history)

In [ ]:
z10, x10 = simulate_regimes(3000, seed=4)
pi10, A10, mu10, sd10, gamma10, h10 = gaussian_hmm_fit(x10, K=2)
calm = int(np.argmin(sd10))                                            # gán nghĩa: chế độ có σ nhỏ = bình lặng
order10 = [calm, 1 - calm]
pred10 = (gamma10.argmax(1) != calm).astype(int)                       # 1 = biến động
acc10 = np.mean(pred10 == z10)
print(f"{len(h10)} vòng | σ học được = {sd10[order10].round(4)} (thật [0.008, 0.025]) | μ = {mu10[order10].round(4)}")
print("A học được:\n", A10[np.ix_(order10, order10)].round(3))
print(f"độ chính xác gán chế độ = {acc10:.3f} | tỉ lệ thời gian biến động thật = {z10.mean():.2f}")

fig, ax = plt.subplots(figsize=(15, 3.5))
ax.plot(x10, lw=0.5, color="k")
ax.fill_between(np.arange(len(x10)), -0.1, 0.1, where=z10 == 1, color="r", alpha=0.15, label="biến động (thật)")
ax.plot(gamma10[:, 1 - calm] * 0.08, color="b", lw=0.8, label="P(biến động) × 0.08")
ax.set_ylim(-0.1, 0.1)
ax.legend(loc="upper right")
plt.show()

In [ ]:
check_true(np.all(np.diff(h10) > -1e-6), "log-likelihood không giảm")
check_close(sd10[order10], [0.008, 0.025], tol=0.003, message="σ của hai chế độ")
check_close(A10[np.ix_(order10, order10)].diagonal(), [0.98, 0.95], tol=0.03, message="xác suất ở lại chế độ")
check_greater(acc10, 0.9, "gán chế độ chính xác > 90%")